In [1]:

import numpy as np
import datetime
from stonesoup.types.array import StateVector, CovarianceMatrix
from stonesoup.types.state import State, GaussianState

from stonesoup.models.transition.linear import (
    CombinedLinearGaussianTransitionModel, ConstantVelocity, KnownTurnRate)
from stonesoup.models.transition.nonlinear import kinematic_bicycle

from stonesoup.simulator.simple import SwitchMultiTargetGroundTruthSimulator, MultiTargetGroundTruthSimulator,  SimpleDetectionSimulator
from stonesoup.types.state import GaussianState
from stonesoup.models.measurement.linear import LinearGaussian
from stonesoup.types.detection import Detection



In [2]:
start_time = datetime.datetime.now()
np.random.seed(42)
initial_state_mean = StateVector([[0], [0], [0], [0]])
initial_state_covariance = CovarianceMatrix(np.diag([4, 0.5, 4, 0.5]))
timestep_size = datetime.timedelta(seconds=1)
number_steps = 50
initial_state = GaussianState(initial_state_mean, initial_state_covariance)


constant_velocity = CombinedLinearGaussianTransitionModel([ConstantVelocity(0.05), ConstantVelocity(0.05)])
turn_left = KnownTurnRate([0.05, 0.05], np.radians(20))
turn_right = KnownTurnRate([0.05, 0.05], np.radians(-20))

model_probs = np.array([[0.7, 0.15, 0.15],  # keep straight, turn left, turn right
                        [0.4, 0.6, 0.0],  # go straight, keep turning left, turn right
                        [0.4, 0.0, 0.6]])  # go straight, turn left, keep turning right

n_truths = 3
xmin = -40
xmax = 40
ymin = -40
ymax = 40
preexisting_states = []

for i in range(0, n_truths):
    x = np.random.randint(xmin, xmax)   # x position of initial state
    y = np.random.randint(ymin, ymax)   # y position of initial state
    y_vel = np.random.randint(-20, 20) / 10  # x velocity will start between -2 and 2
    x_vel = np.random.randint(-20, 20) / 10  # y velocity will start between -2 and 2
    preexisting_states.append(StateVector([x, x_vel, y, y_vel]))

# %%
# Now we have initialised everything, so we can generate the ground truth:
ground_truth_gen = SwitchMultiTargetGroundTruthSimulator(
    initial_state=initial_state,
    transition_models=[constant_velocity, turn_left, turn_right],
    model_probs=model_probs,  # put in matrix from above
    number_steps=number_steps,  # how long we want each track to be
    timestep=timestep_size,
    birth_rate=0,
    death_probability=0,
    preexisting_states=preexisting_states
)


In [3]:
measurement_model = LinearGaussian(
    ndim_state=4,  # Number of state dimensions (position and velocity in 2D)
    mapping=(0, 2),  # Mapping measurement vector index to state index
    noise_covar=np.array([[0.5, 0],  # Covariance matrix for Gaussian PDF
                          [0, 0.5]])
    )


meas_range = np.array([[xmin, xmax], [ymin , ymax]])
detector   = SimpleDetectionSimulator(ground_truth_gen,measurement_model,meas_range=meas_range, detection_probability=1,clutter_rate=0)

In [4]:
detections = set()
ground_truth = set()

for time, dets in detector:
    detections |= dets
    ground_truth |= ground_truth_gen.groundtruth_paths

timesteps = [x.timestamp for x in ground_truth[0]]

from stonesoup.plotter import AnimatedPlotterly

plotter = AnimatedPlotterly(timesteps=timesteps)
plotter.plot_ground_truths(ground_truth, [0, 2])
plotter.plot_measurements(detections, [0, 2])
plotter.fig

In [ ]:
from stonesoup.predictor.particle import ParticlePredictor
from stonesoup.resampler.particle import ESSResampler
from stonesoup.updater.particle import ParticleUpdater
from stonesoup.hypothesiser.distance import DistanceHypothesiser
from stonesoup.measures import Mahalanobis
from stonesoup.dataassociator.neighbour import GNNWith2DAssignment
from stonesoup.deleter.time import UpdateTimeDeleter
from stonesoup.initiator.simple import GaussianParticleInitiator
from stonesoup.types.state import GaussianState
from stonesoup.initiator.simple import SimpleMeasurementInitiator
from stonesoup.tracker.simple import MultiTargetTracker
from stonesoup.models.transition.nonlinear import kinematic_bicycle

measurement_model = LinearGaussian(ndim_state=5,mapping=(0, 1), noise_covar=np.array([[1, 0], [0, 1]]))

#transition_model_estimate = CombinedLinearGaussianTransitionModel([ConstantVelocity(5),ConstantVelocity(5)])
transition_model_estimate = kinematic_bicycle(std_accl=1, std_accd=1,L=1)
predictor_PF              = ParticlePredictor(transition_model_estimate)
resampler                 = ESSResampler()
updater_PF                = ParticleUpdater(measurement_model=measurement_model, resampler=resampler)
hypothesiser_PF           = DistanceHypothesiser(predictor_PF, updater_PF, measure=Mahalanobis(), missed_distance=20)
data_associator_PF        = GNNWith2DAssignment(hypothesiser_PF)
deleter                   = UpdateTimeDeleter(datetime.timedelta(seconds=5), delete_last_pred=True)
prior_state               = GaussianState(StateVector([0, 0, 0, 0, 0]), np.diag([10, 10, np.pi/2, 10, np.pi/2]) ** 2)
initiator_Part            = SimpleMeasurementInitiator(prior_state, measurement_model=measurement_model, skip_non_reversible=True)
initiator_PF              = GaussianParticleInitiator(number_particles=1000,initiator=initiator_Part,use_fixed_covar=False)
tracker_PF                = MultiTargetTracker(initiator=initiator_PF, deleter=deleter,detector=detector,data_associator=data_associator_PF,updater=updater_PF)

tracks_PF = set()
for step, (time, current_tracks) in enumerate(tracker_PF, 1):
    tracks_PF.update(current_tracks)

plotter = AnimatedPlotterly(timesteps=timesteps)
plotter.plot_measurements(detections, [0, 1])
plotter.plot_ground_truths(ground_truth, [0, 1])
plotter.plot_tracks(tracks_PF, [0, 1], label="PF", particle=True, plot_history=False)
plotter.fig



ValueError: operands could not be broadcast together with shapes (4,4) (5,5) 

In [ ]:
current_tracks.pop()

Track(
    states=[ParticleStateUpdate(
               state_vector=StateVectors([[-16.52279808, -16.6728312 , -14.82692991, ...,
                                           -16.3236542 , -15.01671082, -16.24060165],
                                          [ -4.78574692,   6.61369463,   3.42963532, ...,
                                             9.68020455,   2.3565075 ,  -8.30155555],
                                          [-38.51401596, -37.7378283 , -37.45619484, ...,
                                           -38.26384863, -38.63459641, -38.41686751],
                                          [ 12.65315981,  -1.61198022,   8.74744083, ...,
                                            -8.48030763,  -3.69832442,  -5.44496354]]),
               hypothesis=SingleHypothesis(
                              prediction=None,
                              measurement=TrueDetection(
                                              state_vector=StateVector([[-16.19844264],
                  



# %%
# 4) Calculate and display metrics to show effectiveness of different tracking algorithms
# ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
#
# The final part of this example is to calculate metrics that can determine how well each tracking
# algorithm followed the target. None will be perfect due to the sensor measurement noise and error
# in data association where multiple tracks meet, but some will perform better than others.
# 
# This section of the example follows code from the metrics example, which is also used in
# the sensor management tutorials. More complete documentation can be found there.
# 
# Firstly, we calculate the Optimal Sub-Pattern Assignment (OSPA) distance at each time
# step for each tracker. This is a measure of how far the calculated tracks are
# from the ground truth. We first initialise the metrics before plotting:

tracking_filters = ['EKF', 'UKF', 'PF', 'ESIF']

# %%
from stonesoup.metricgenerator.ospametric import OSPAMetric

ospa_generators = [OSPAMetric(c=40, p=1,
                              generator_name=f'{tracking_filter} OSPA metrics',
                              tracks_key=f'tracks_{tracking_filter}',
                              truths_key='truths'
                             )
                   for tracking_filter in tracking_filters]

from stonesoup.metricgenerator.tracktotruthmetrics import SIAPMetrics
from stonesoup.measures import Euclidean

siap_generators = [SIAPMetrics(position_measure=Euclidean((0, 2)),
                             velocity_measure=Euclidean((1, 3)),
                             generator_name=f'{tracking_filter} SIAP metrics',
                             tracks_key=f'tracks_{tracking_filter}',
                             truths_key='truths'
                            )
                  for tracking_filter in tracking_filters]


from stonesoup.metricgenerator.uncertaintymetric import SumofCovarianceNormsMetric

uncertainty_generators = [
    SumofCovarianceNormsMetric(generator_name=f'{tracking_filter} OSPA metrics',
                               tracks_key=f'tracks_{tracking_filter}')
    for tracking_filter in tracking_filters]

# %%
# Now we initialise the metric manager and generate the metrics:

from stonesoup.dataassociator.tracktotrack import TrackToTruth
from stonesoup.metricgenerator.manager import MultiManager

associator = TrackToTruth(association_threshold=30)

generators = ospa_generators + siap_generators + uncertainty_generators
metric_manager = MultiManager(generators, associator=associator)

metric_manager.add_data({'truths': ground_truth,
                         'tracks_EKF': tracks_EKF,
                         'tracks_UKF': tracks_UKF,
                         'tracks_PF': tracks_PF,
                         'tracks_ESIF': tracks_ESIF
                         })
metrics = metric_manager.generate_metrics()

# %%
# Now we can plot the OSPA distance for each tracker:

from stonesoup.plotter import MetricPlotter

fig1 = MetricPlotter()
fig1.plot_metrics(metrics, metric_names=['OSPA distances'])

# %%
# It can be seen that the EKF, UKF, and Particle Filter all behave very similarly,
# whereas the ESIF has very poor relative performance. A singular performance metric is calculated
# from this by summing the OSPA value over all timesteps:

# sum up distance error from ground truth over all timestamps
for tracking_filter in tracking_filters:
    total = sum([metrics[f'{tracking_filter} OSPA metrics']['OSPA distances'].value[i].value
                 for i in range(0, len(metrics[f'{tracking_filter} OSPA metrics']['OSPA distances'].value))])
    print(f'OSPA total value for {tracking_filter} is {total:.3f}')

# %%
# Finally, we calculate the SIAP metrics for the EKF. The same metrics can be calculated for the
# other trackers if desired. The user can copy this section of code and replace the relevant
# variable names to get the full metrics for the UKF, ESIF, and Particle Filter.
from stonesoup.metricgenerator.metrictables import SIAPTableGenerator

# generate metrics for EKF
siap_metrics = metrics['EKF SIAP metrics']
siap_averages_EKF = {siap_metrics.get(metric) for metric in siap_metrics
                     if metric.startswith("SIAP") and not metric.endswith(" at times")}

_ = SIAPTableGenerator(siap_averages_EKF).compute_metric()
print("\n\nSIAP metrics for EKF:")
